In [ ]:
import random
from datasets import load_dataset, Dataset
import time

# =============================================================================
# 🧠 AI 튜터 모드 ON! 🧑‍🏫
# 🌟 환영합니다! 우리는 지금 '한국 수능 기출 문제 분석기'를 만들어볼 거예요. 🌟
# 📖 이 데이터셋은 'cfpark00/KoreanSAT'으로, 실제 대학수학능력시험(SAT)의 수학 문제와 정답이 담겨있어요.
# 📚 목표는 단순한 데이터 탐색을 넘어, 문제(problem) 텍스트를 분석해서 어떤 유형의 문제를 받았는지 추론하고,
# 🤖 간단한 AI 모델이 데이터를 어떻게 학습하는지 경험해보는 거예요. 걱정 마세요, 코드가 길어도 같이 보면 금방 이해할 거예요!
# =============================================================================

# --- [설정 변수] ---
DATASET_NAME = "cfpark00/KoreanSAT"
TARGET_SPLIT = '2023_math' # 원하는 특정 세트를 지정합니다.
SAMPLE_COUNT = 10 # 테스트를 위해 상위 10개만 사용합니다! 메모리 절약 필수!

print("=" * 80)
print("✨ [미션 시작] 한국 수능 기출 문제 데이터셋 탐험 임무를 시작합니다! ✨")
print("=" * 80)

# 1. 데이터 로드 (스트리밍을 통한 효율적인 로딩 시도)
dataset = None
try:
    print(f"[Step 1/3] 💾 데이터를 스트리밍 모드(streaming=True)로 로드하며 속도를 확인합니다...")
    start_time = time.time()
    # Streaming=True로 시도합니다. (불필요한 다운로드 최소화)
    dataset = load_dataset(DATASET_NAME, split=TARGET_SPLIT, streaming=True)
    print(f"🎉 성공! 스트리밍 로드가 완료되었습니다. 소요 시간: {time.time() - start_time:.2f}초")

except Exception as e:
    # 스트리밍이 안 되거나 네트워크 문제 발생 시 예외 처리
    print(f"⚠️ 스트리밍 모드 로드에 실패했거나 제약이 있습니다: {e}")
    print("   => 안전을 위해, 기본 Dataset 모드로 로드하여 임시로 진행합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split=TARGET_SPLIT, streaming=False)
        print("✅ 일반 Dataset 모드로 로드 성공! 이제 마음껏 데이터를 탐색할 수 있어요.")
    except Exception as e_fallback:
        print(f"❌ 치명적인 오류: 데이터셋을 로드할 수 없습니다. 오류 내용: {e_fallback}")
        exit()

# 2. 샘플링 및 데이터 준비 (Iteration 기반 처리)
print("\n" + "=" * 80)
print(f"[Step 2/3] 🛠️ {SAMPLE_COUNT}개의 핵심 샘플을 뽑아 실습 환경을 조성합니다...")

# 필수 규칙에 따라 .take()를 사용하고 list()로 변환합니다.
sampled_dataset = list(dataset.take(SAMPLE_COUNT))
sample_data_list = sampled_dataset

print(f"✨ 분석에 사용할 샘플 데이터 크기: 총 {len(sample_data_list)}개의 '핵심 문제'를 확보했습니다!")

# 3. 데이터 구조 파악 및 분석
print("\n" + "=" * 80)
print("✨ [Step 3/3] 🔎 데이터 구조 파악 및 핵심 Feature 분석 시간입니다!")

# 첫 번째 샘플을 보고 데이터의 컬럼(Feature)을 확인합니다.
if sample_data_list:
    first_sample = sample_data_list[0]
    print("\n--- 💖 첫 번째 문제 샘플 미리보기 ---")
    print(f"  [ID]       : {first_sample['id']}")
    print(f"  [문제 이름] : {first_sample['name']} (문제 분류 유형)")
    print(f"  [문제 내용] : {first_sample['problem'][:50]}...") # 문제 내용 처음 50자만 출력
    print(f"  [정답]     : {first_sample['answer']}")
    print(f"  [점수]     : {first_sample['score']}")
    print("----------------------------------")

    # =======================================================================
    # 🌟 [창의적 실습 1]: 문제 텍스트에서 '핵심 키워드' 추출하기 (NLP 기초 시뮬레이션)
    # -----------------------------------------------------------------------
    print("\n\n▶️ [실습 1] 💬 문제 텍스트에서 '키워드'를 뽑아내어 '분류'하는 시뮬레이션")
    print("   (Goal: 복잡한 문장 속에서 중요한 숫자나 단어를 찾는 능력 훈련)")
    
    def extract_keyword(sample):
        """문제 텍스트(problem)에서 단순 패턴(키워드/숫자)을 찾아봅니다."""
        problem = sample['problem']
        
        # 만약 '확률'이나 '통계' 같은 단어가 포함되어 있으면 해당 키워드를 반환
        if '확률' in problem or '통계' in problem:
            return "확률/통계 키워드 감지!"
        # '사각형'이나 '직선' 같은 기하학적 단어가 있다면 해당 키워드를 반환
        elif '사각형' in problem or '도형' in problem:
            return "기하학적 구조물 감지!"
        else:
            return "일반 유형의 수학 문제"

    keywords_list = []
    for sample in sample_data_list:
        keywords = extract_keyword(sample)
        keywords_list.append(keywords)
        
    print("   [결과 확인] ---")
    print(f"   {len(keywords_list)}개 문제 분석 완료. 감지된 키워드 분포:")
    from collections import Counter
    print(Counter(keywords_list))


    # =======================================================================
    # 📊 [창의적 실습 2]: 연도별/유형별 평균 점수 분석 (정량적 분석)
    # -----------------------------------------------------------------------
    print("\n\n▶️ [실습 2] 📈 데이터의 성능을 분석! 평균 점수 확인하기")
    print("   (Goal: 특정 카테고리(예: 2022년 vs 2023년)의 성능 차이를 분석)")
    
    # 이 샘플은 특정 연도('2023_math')로 제한되어 있으므로, 이름으로 그룹화는 어려우나
    # 만약 전체 데이터가 로드되었다면 이렇게 처리할 수 있습니다.
    
    total_score = 0
    print(f"   [분석] {TARGET_SPLIT}년도 {len(sample_data_list)}개 샘플의 평균 점수를 계산합니다...")

    for i, sample in enumerate(sample_data_list):
        total_score += sample['score']
        # 초보자에게 재미를 주기 위해 10번째마다 진행 상황 출력
        if (i + 1) % 5 == 0 and i < len(sample_data_list) - 1:
            print(f"   ... {i+1}번째 문제 분석 완료. 현재까지 합계 점수: {total_score}")

    average_score = total_score / len(sample_data_list)
    
    print("\n-----------------------------------------")
    print(f"🎉 최종 결과: 분석된 {len(sample_data_list)}개 샘플의 평균 점수는 {average_score:.2f}점입니다.")
    print("=> 이 점수가 이 시험의 난이도를 추측하는 데 도움이 될 수 있겠죠?")

    # =======================================================================
    # 💡 [마무리 팁] 🏆 실제 AI 학습 시 고려할 점
    # =======================================================================
    print("\n\n================================================================================================")
    print("✨ 튜터의 최종 코칭: 이 데이터를 실제 AI로 활용하려면?")
    print("1. 문제(problem) 텍스트를 전처리(Pre-process)해야 합니다. (불필요한 특수문자, 단위 제거)")
    print("2. 문제의 구조화(Structured QA)가 중요합니다. '질문'과 '답변'을 명확히 분리하는 전용 레이블링이 필요합니다.")
    print("3. 'review' 필드는 학생의 해설이나 오답 패턴을 담고 있어, 사용자 피드백(Feedback) 시스템에 활용하기 아주 좋습니다! ✍️")
    print("================================================================================================")
else:
    print("\n🚨 경고: 데이터를 로드하거나 샘플링하는 과정에서 문제가 발생했습니다. 코드를 확인해주세요.")